In [1]:
!pip install openai -q

In [2]:
import os
import requests
from openai import OpenAI
from typing import List, Optional
from pydantic import BaseModel, Field

In [3]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY']=OPENAI_API_KEY

In [4]:
def render_transcript(d: dict) -> str:
    """
    Renders the transcript data from a dictionary into a formatted string.

    Args:
        d: A dictionary containing the transcript data. Expected keys are 'symbol',
           'quarter', and 'transcript'. The 'transcript' key should contain a list
           of dictionaries, each with 'speaker', 'title' (optional), and 'content'.

    Returns:
        A formatted string representation of the transcript.
    """
    header_info = f"Symbol: {d.get('symbol','')} | Quarter: {d.get('quarter','')}"
    transcript_lines = [header_info, "TRANSCRIPT:"]
    for index, transcript_entry in enumerate(d.get("transcript", [])):
        speaker_name = transcript_entry.get("speaker", "Unknown")
        speaker_title = transcript_entry.get("title")
        utterance_text = (transcript_entry.get("content") or "").strip()
        speaker_tag = f"{speaker_name} ({speaker_title})" if speaker_title else speaker_name
        transcript_lines.append(f"[{index:04d}] {speaker_tag}: {utterance_text}")
    return "\n".join(transcript_lines)

In [5]:
class EarningsCallInsights(BaseModel):
    """
    Output estructurado con insights clave en español extraídos de una earnings call.
    """
    symbol: Optional[str] = Field(description="Ticker de la compañía, ej. AAPL, AMZN, MSFT, etc")
    sentiment: Optional[str] = Field(description="Sentimiento general: Muy bajista / Bajista / Alcista / Muy alcista")
    summary: str = Field(description="Resumen ejecutivo de la llamada en español (párrafo de 3 líneas)")
    key_topics: List[str] = Field(default_factory=list, description="Temas estratégicos principales, cado uno con un máximo de 4 palabras")
    guidance: List[str] = Field(default_factory=list, description="Guías o proyecciones futuras mencionadas, en español")
    numeric_highlights: List[str] = Field(default_factory=list, description="Cifras o métricas clave reportadas, en español")
    risks: List[str] = Field(default_factory=list, description="Riesgos explícitos o implícitos discutidos, en español")
    catalysts: List[str] = Field(default_factory=list, description="Catalizadores futuros o eventos relevantes, en español")
    analyst_questions: List[str] = Field(default_factory=list, description="Top 3 preguntas destacadas de analistas, en español")
    unanswered_topics: List[str] = Field(default_factory=list, description="Temas abiertos o sin respuesta clara, en español")
    bullish_points: List[str] = Field(default_factory=list, description="Tesis alcistas derivadas de la llamada, en español")
    bearish_points: List[str] = Field(default_factory=list, description="Tesis bajistas derivadas de la llamada, en español")
    red_flags: List[str] = Field(default_factory=list, description="Alertas o señales negativas detectadas, en español")
    emotion: Optional[str] = Field(description="Emoción general: Optimismo / Incertidumbre / Preocupación / Entusiasmo / Frustración")

In [6]:
def get_earnings_call_insights(call_transcripts: dict) -> EarningsCallInsights:
    """
    Obtiene insights clave de una transcripción de earnings call utilizando un modelo de OpenAI.

    Args:
        transcript_text: El texto de la transcripción de la llamada.

    Returns:
        Un objeto diccionario derivado de EarningsCallInsights con los insights extraídos.
    """
    client = OpenAI()
    transcript_text = render_transcript(call_transcripts)
    response = client.chat.completions.parse(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": "Eres un experto Analista Financiero. Devuelve SOLO un JSON válido que siga exactamente el esquema de EarningsCallInsights. Salidas en español."},
            {"role": "user", "content": transcript_text},
        ],
        response_format=EarningsCallInsights,
    )
    insights = response.choices[0].message.parsed
    return insights.model_dump()

In [7]:
def get_call_transcripts(ticker: str, quarter: str) -> dict:
    client = OpenAI()
    alphavantage_api = userdata.get('ALPHAVANTAGE_API')
    url_transcripts = f'https://www.alphavantage.co/query?function=EARNINGS_CALL_TRANSCRIPT&symbol={ticker}&quarter={quarter}&apikey={alphavantage_api}'
    response_transcripts = requests.get(url_transcripts)
    call_transcripts = response_transcripts.json()
    insights = get_earnings_call_insights(call_transcripts=call_transcripts)
    return insights

In [8]:
insights = get_call_transcripts(ticker="NVDA", quarter="2026Q2")

In [9]:
insights

{'symbol': 'NVDA',
 'sentiment': 'Muy alcista',
 'summary': 'NVIDIA reportó un trimestre récord con ingresos totales de $46.7B y un crecimiento secuencial en todas las plataformas, impulsado por una fuerte adopción de la plataforma Blackwell y ramp de GB300/GV300.\nLa compañía guió sólido para Q3 ($54B ±2%) y reafirmó Rubin en producción en 2026, mientras destaca oportunidades de $3–4T en inversión de infraestructura AI hacia el fin de la década.\nRiesgos geopolíticos (H20/China), un aumento de inventario a $15B y mayores gastos operativos introducen incertidumbre táctica pese al impulso operativo y tecnológico.',
 'key_topics': ['Blackwell',
  'Rubin',
  'GB300 / GV300',
  'NVLink 72',
  'H20 y China',
  'AI infraestructura $3-4T',
  'Spectrum X / XGS',
  'Redes y InfiniBand',
  'RTX / GeForce NOW'],
 'guidance': ['Ingresos Q3 previstos $54B ±2% (no incluye H20 a China)',
  'Margen bruto GAAP esperado 73.3% (±50 pb); non-GAAP 73.5% (±50 pb)',
  'Gastos operativos Q3: GAAP ≈ $5.9B; non

### Function Calling

In [10]:
import json
import gradio as gr

In [11]:
get_call_transcripts_json = {
    "name": "get_call_transcripts",
    "description": "Usa esta herramienta (tool) para obtener insights clave de una transcripción de earnings call",
    "parameters": {
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "El TICKER de la compañía a analizar."
            },
            "quarter": {
                "type": "string",
                "description": "El trimestre de transcripciones de la llamada con Inversionistas"
            }
        },
        "required": ["ticker", "quarter"],
        "additionalProperties": False
    }
}

In [12]:
tools = [{"type": "function", "function": get_call_transcripts_json}]
tools

[{'type': 'function',
  'function': {'name': 'get_call_transcripts',
   'description': 'Usa esta herramienta (tool) para obtener insights clave de una transcripción de earnings call',
   'parameters': {'type': 'object',
    'properties': {'ticker': {'type': 'string',
      'description': 'El TICKER de la compañía a analizar.'},
     'quarter': {'type': 'string',
      'description': 'El trimestre de transcripciones de la llamada con Inversionistas'}},
    'required': ['ticker', 'quarter'],
    'additionalProperties': False}}}]

In [13]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [14]:
system_prompt = "Eres un experto en inversiones"

In [17]:
def chat(message, history):
    client = OpenAI()
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = client.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason

        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [18]:
gr.ChatInterface(chat, type="messages").launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://29791d7fd7637ead9c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Tool called: get_call_transcripts
Tool called: get_call_transcripts
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://29791d7fd7637ead9c.gradio.live


Ejemplos:

* Insights de la llamada con inversionistas de AMZN del 2025Q3
* Insights de la llamada con inversionistas de GOOGL del 2025Q3
* Insights de la llamada con inversionistas del NVDA del 2025Q2
* Insights de la llamada con inversionistas de AMD del 2025Q2